[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microscopy-processing/N2N-REO/blob/main/N2N-EO.ipynb)

# N2N-EO (Noise2Noise Even Odd denoising)

## Packages

In [ ]:
# pip install mrcfile
import mrcfile

In [ ]:
# pip install numpy
import numpy as np

In [ ]:
# pip install tqdm ipywidgets
from tqdm.notebook import tqdm

In [ ]:
# pip install matplotlib
import matplotlib.pyplot as plt

In [ ]:
# pip install opencv-python
import cv2

In [ ]:
import json

In [ ]:
# pip install tensorflow

In [ ]:
# pip install cryoCARE --no-deps

In [ ]:
# pip install csbdeep

In [ ]:
# pip install gdown
import gdown

## Download a noisy tomogram

In [ ]:
#url="https://drive.google.com/file/d/1UtFnrgTj0JE0wF3GfPNF8BBWat83j0U_/view?usp=drive_link
url="https://drive.google.com/file/d/1DZzmeJtso8woza4bPgQfUezBnTZrufuL/view?usp=sharing"
gdown.download(url, output="noisy_vol.mrc", quiet=False, use_cookies=False)

In [ ]:
!ls -l noisy_vol.mrc

In [ ]:
X = mrcfile.open("noisy_vol.mrc").data

In [ ]:
X.shape

## Split the tomogram in even and odd axial slices

In [ ]:
even_vol = X[0::2,:,:]
with mrcfile.new("even.mrc", overwrite=True) as mrc:
    mrc.set_data(even_vol)
    mrc.data

In [ ]:
even_vol.shape

In [ ]:
!ls -l "even.mrc"

In [ ]:
odd_vol = X[1::2,:,:]
with mrcfile.new("odd.mrc", overwrite=True) as mrc:
    mrc.set_data(odd_vol)
    mrc.data
    mrc.data

In [ ]:
odd_vol.shape

In [ ]:
!ls -l "odd.mrc"

In [ ]:
slice_idx = even_vol.shape[0] // 2

fig, axes = plt.subplots(1, 3, figsize=(20, 20))

im1 = axes[0].imshow(even_vol[slice_idx, ...].T, cmap='gray', origin='lower')
axes[0].set_title(f'Original Slice Even z={slice_idx}')
axes[0].grid(False)

im2 = axes[1].imshow(odd_vol[slice_idx, ...].T, cmap='gray', origin='lower')
axes[1].set_title(f'Original Slice Odd z={slice_idx}')
axes[1].grid(False)

im3 = axes[2].imshow((even_vol[slice_idx, ...].T - odd_vol[slice_idx, ...].T + 128).astype(np.int16), cmap='gray', origin='lower')
axes[2].set_title(f'even[z] - odd[z]')
axes[2].grid(False)

plt.tight_layout()
plt.show()

# Denoising

In [ ]:
_ = {
    "even": ["even.mrc"],
    "odd": ["odd.mrc"],
    "mask": [""],
    "patch_shape": [8, 8, 8], # <- Be careful here: in this example the tomogram is very small and the patch shape must be also small
    "num_slices": 800,
    "split": 0.9,
    "tilt_axis": "Y",
    "n_normalization_samples": 200,
    "path": "./data_EO",
    "overwrite": "True"  
}

with open("train_data_config__EO.json", 'w') as f:
    json.dump(_, f, indent=4)

In [ ]:
%%bash
cryoCARE_extract_train_data.py --conf train_data_config__EO.json

In [ ]:
%%writefile train_config__EO.json
{
  "train_data": "./data_EO",
  "epochs": 50,
  "steps_per_epoch": 200,
  "batch_size": 16,
  "unet_kern_size": 3,
  "unet_n_depth": 3,
  "unet_n_first": 16,
  "learning_rate": 0.0004,
  "model_name": "model_EO",
  "path": "./",
  "gpu_id": [0]
}

In [ ]:
%%bash
cryoCARE_train.py --conf train_config__EO.json

In [ ]:
_ = {
    "path": "./model_EO.tar.gz",
    "even": ["noisy_vol.mrc"], 
    "odd": ["noisy_vol.mrc"],
    "n_tiles": [1,1,1],
    "output": "denoised_vol_EO",
    "overwrite": "True",
    "gpu_id": [0]
}

with open("predict_config__EO.json", 'w') as f:
    json.dump(_, f, indent=4)

In [ ]:
%%bash
cryoCARE_predict.py --conf predict_config__EO.json || true

In [ ]:
!ls -l denoised_vol_EO/noisy_vol.mrc

In [ ]:
Y = mrcfile.read("denoised_vol_EO/noisy_vol.mrc").astype(np.int16)

In [ ]:
X.shape

In [ ]:
Y.shape

In [ ]:
slice_idx = X.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(15, 15))

# Plot a original slice
im1 = axes[0].imshow(X[slice_idx, :, :].T, cmap='gray', origin='lower')
axes[0].set_title(f'Original Slice Z={slice_idx}')
axes[0].grid(False)

# Plot a denoised slice
im2 = axes[1].imshow(Y[slice_idx, :, :].T, cmap='gray', origin='lower')
axes[1].set_title(f'N2N-Odd-Even Denoised Slice Z={slice_idx}')
axes[1].grid(False)

plt.tight_layout()
plt.show()